# S02 · Jerarquía de memoria y *tiling* — práctica guiada (Colab/Kaggle)

> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Sesión S02 · martes 24 de noviembre de 2026 · 08:30–10:00 · Lab LS4

**Antes de ejecutar:** en Colab, *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. En Kaggle, *Settings → Accelerator → GPU*.
Si la primera celda no muestra una GPU, no sigas: nada de lo que sigue correrá.

Protocolo de medición del curso: anota la **GPU asignada y la fecha** (cambian entre sesiones), cada programa hace 1 calentamiento + 5 repeticiones y reporta la **mediana**, y las copias anfitrión↔dispositivo se reportan aparte.
Los tiempos que aparecen en el README son órdenes de magnitud para una T4; los tuyos dependen de la GPU que te toque.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## Paso 1 · Escribir los fuentes en el disco de la máquina virtual

Cada celda `%%writefile` crea un archivo. Son los mismos archivos de `practica-guiada/` del repositorio.

In [ ]:
%%writefile common.cuh
// common.cuh — utilidades compartidas por matmul_naive.cu, matmul_tiled.cu y transpose.cu.
// Contiene: verificación de errores de CUDA, inicialización determinista de matrices,
// referencia secuencial en CPU, comparación con tolerancia, mediana y lectura de argumentos.
// Nada de aquí corre en la GPU: todo es código de anfitrión (host).
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

// Toda llamada a la API de CUDA se envuelve en CUDA_CHECK. Si falla, imprime archivo, línea
// y mensaje, y termina. Sin esto, un error pasa en silencio y el programa "parece" funcionar.
#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err__ = (call);                                               \
        if (err__ != cudaSuccess) {                                               \
            fprintf(stderr, "Error CUDA en %s:%d: %s\n", __FILE__, __LINE__,      \
                    cudaGetErrorString(err__));                                   \
            exit(EXIT_FAILURE);                                                   \
        }                                                                         \
    } while (0)

// Número de repeticiones cronometradas (protocolo de medición: calentamiento + 5, mediana).
#define REPS 5

// Inicialización determinista. La MISMA fórmula está en matmul_cpu.c y matmul_numba.py,
// para que las sumas de comprobación (checksum) sean comparables entre programas.
static inline void init_matrices(float *A, float *B, int n) {
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < n; ++j) {
            A[(size_t)i * n + j] = (float)((3 * i + 7 * j) % 11) / 11.0f;
            B[(size_t)i * n + j] = (float)((5 * i + 2 * j) % 13) / 13.0f;
        }
    }
}

// Referencia secuencial. Orden i-k-j: el ciclo interno recorre B y C por filas (amigable con
// la caché de la CPU). Para cada C[i][j] la suma sobre k va en orden creciente, igual que en
// los kernels, así que la diferencia con la GPU es solo redondeo (FMA, orden de operaciones).
static inline void matmul_cpu(const float *A, const float *B, float *C, int n) {
    for (size_t idx = 0; idx < (size_t)n * n; ++idx) C[idx] = 0.0f;
    for (int i = 0; i < n; ++i) {
        for (int k = 0; k < n; ++k) {
            float a = A[(size_t)i * n + k];
            for (int j = 0; j < n; ++j) {
                C[(size_t)i * n + j] += a * B[(size_t)k * n + j];
            }
        }
    }
}

// Tolerancia de comparación. Los sumandos están en [0, 1) y cada C[i][j] vale ~0.2·n; el error de
// redondeo en float (FMA en la GPU, suma simple en la CPU) crece con n y queda muy por debajo de
// 1e-4 * n. Esa tolerancia detecta de sobra un elemento no calculado (error ~0.2·n) o basura.
static inline float tolerance(int n) { return 1e-4f * (float)n; }

// Verifica C contra la CPU. Para n <= 1024 compara TODOS los elementos. Para n mayor, la
// referencia completa tardaría demasiado en el anfitrión, así que compara una muestra
// determinista de 4096 elementos (cada uno recalculado con su propio producto punto).
// Devuelve cuántos elementos comparó y deja el error máximo en *max_err.
static inline long verify_matmul(const float *A, const float *B, const float *C, int n,
                          float *max_err) {
    *max_err = 0.0f;
    if (n <= 1024) {
        float *ref = (float *)malloc((size_t)n * n * sizeof(float));
        if (ref == NULL) { fprintf(stderr, "Sin memoria para la referencia\n"); exit(EXIT_FAILURE); }
        matmul_cpu(A, B, ref, n);
        for (size_t idx = 0; idx < (size_t)n * n; ++idx) {
            float e = fabsf(ref[idx] - C[idx]);
            if (e > *max_err) *max_err = e;
        }
        free(ref);
        return (long)n * n;
    }
    const int samples = 4096;
    unsigned long long state = 12345ULL;   // generador congruencial: misma muestra siempre
    for (int s = 0; s < samples; ++s) {
        state = state * 6364136223846793005ULL + 1442695040888963407ULL;
        int i = (int)((state >> 33) % (unsigned long long)n);
        state = state * 6364136223846793005ULL + 1442695040888963407ULL;
        int j = (int)((state >> 33) % (unsigned long long)n);
        float acc = 0.0f;
        for (int k = 0; k < n; ++k) acc += A[(size_t)i * n + k] * B[(size_t)k * n + j];
        float e = fabsf(acc - C[(size_t)i * n + j]);
        if (e > *max_err) *max_err = e;
    }
    return samples;
}

// Mediana de k valores (k pequeño): ordenamiento por inserción y elemento central.
static inline float median_of(float *v, int k) {
    for (int i = 1; i < k; ++i) {
        float x = v[i];
        int j = i - 1;
        while (j >= 0 && v[j] > x) { v[j + 1] = v[j]; --j; }
        v[j + 1] = x;
    }
    return v[k / 2];
}

static inline float min_of(const float *v, int k) {
    float m = v[0];
    for (int i = 1; i < k; ++i) if (v[i] < m) m = v[i];
    return m;
}

static inline float max_of(const float *v, int k) {
    float m = v[0];
    for (int i = 1; i < k; ++i) if (v[i] > m) m = v[i];
    return m;
}

// Tamaño del problema por línea de comandos, con valor por defecto.
static inline int parse_n(int argc, char **argv, int default_n) {
    int n = default_n;
    if (argc > 1) n = atoi(argv[1]);
    if (n <= 0) {
        fprintf(stderr, "Uso: %s [n]   (n > 0; por defecto %d)\n", argv[0], default_n);
        exit(EXIT_FAILURE);
    }
    return n;
}

In [ ]:
%%writefile matmul_naive.cu
// matmul_naive.cu — multiplicación de matrices C = A·B (n×n, float, por filas) en CUDA.
// Versión INGENUA: un hilo por elemento de C; cada hilo lee una fila completa de A y una
// columna completa de B directamente desde memoria GLOBAL.
//
// Compilar (GPU):        nvcc -O2 -arch=sm_75 matmul_naive.cu -o matmul_naive
// Verificar sin GPU:     nvcc -c -arch=sm_75 matmul_naive.cu -o matmul_naive.o
// Ejecutar:              ./matmul_naive [n]        (n por defecto: 512)
// Tamaño de bloque:      -DBLOCK=8 | 16 | 32       (por defecto 16 → bloques de 16×16 = 256 hilos)
#include "common.cuh"

#ifndef BLOCK
#define BLOCK 16
#endif

// Un hilo calcula C[row][col]. Las lecturas de A[row][k] las comparten los n hilos de la
// misma fila; las de B[k][col], los n hilos de la misma columna. Ningún dato se reutiliza
// dentro del bloque: cada elemento de A y de B se lee n veces desde memoria global.
__global__ void matmul_naive(const float *A, const float *B, float *C, int n) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // x recorre columnas (direcciones consecutivas)
    int row = blockIdx.y * blockDim.y + threadIdx.y;   // y recorre filas
    if (row < n && col < n) {                           // guardia: la malla puede sobrar
        float acc = 0.0f;
        for (int k = 0; k < n; ++k) {
            acc += A[row * n + k] * B[k * n + col];     // 2 lecturas globales por iteración
        }
        C[row * n + col] = acc;
    }
}

int main(int argc, char **argv) {
    int n = parse_n(argc, argv, 512);
    size_t bytes = (size_t)n * n * sizeof(float);

    // --- Anfitrión: reservar e inicializar ---
    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C = (float *)malloc(bytes);
    if (!h_A || !h_B || !h_C) { fprintf(stderr, "Sin memoria en el anfitrión\n"); return 1; }
    init_matrices(h_A, h_B, n);

    // --- Dispositivo: reservar ---
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));

    cudaEvent_t e_start, e_stop;
    CUDA_CHECK(cudaEventCreate(&e_start));
    CUDA_CHECK(cudaEventCreate(&e_stop));

    // --- Copias anfitrión → dispositivo (se cronometran: las copias cuentan) ---
    CUDA_CHECK(cudaEventRecord(e_start));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(e_stop));
    CUDA_CHECK(cudaEventSynchronize(e_stop));
    float h2d_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&h2d_ms, e_start, e_stop));

    // C en ceros: si algún hilo no escribe su elemento, la verificación lo detecta.
    CUDA_CHECK(cudaMemset(d_C, 0, bytes));

    // --- Configuración de lanzamiento: malla 2D de bloques 2D ---
    dim3 block(BLOCK, BLOCK);
    dim3 grid((n + BLOCK - 1) / BLOCK, (n + BLOCK - 1) / BLOCK);   // redondeo hacia arriba

    // Calentamiento (no se cuenta) + revisión de errores de lanzamiento y de ejecución.
    matmul_naive<<<grid, block>>>(d_A, d_B, d_C, n);
    CUDA_CHECK(cudaGetLastError());          // errores de configuración (p. ej. bloque demasiado grande)
    CUDA_CHECK(cudaDeviceSynchronize());     // errores ocurridos durante la ejecución del kernel

    // --- Medición: REPS repeticiones, se reporta la mediana ---
    float t_ms[REPS];
    for (int r = 0; r < REPS; ++r) {
        CUDA_CHECK(cudaEventRecord(e_start));
        matmul_naive<<<grid, block>>>(d_A, d_B, d_C, n);
        CUDA_CHECK(cudaEventRecord(e_stop));
        CUDA_CHECK(cudaEventSynchronize(e_stop));
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventElapsedTime(&t_ms[r], e_start, e_stop));
    }

    // --- Copia dispositivo → anfitrión ---
    CUDA_CHECK(cudaEventRecord(e_start));
    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(e_stop));
    CUDA_CHECK(cudaEventSynchronize(e_stop));
    float d2h_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&d2h_ms, e_start, e_stop));

    // --- Verificación contra la CPU: "no dio error" no significa "está bien" ---
    float max_err = 0.0f;
    long checked = verify_matmul(h_A, h_B, h_C, n, &max_err);
    int ok = (max_err <= tolerance(n));

    // --- Reporte ---
    float med = median_of(t_ms, REPS);        // ordena t_ms; min/max se toman después
    float mn = min_of(t_ms, REPS), mx = max_of(t_ms, REPS);
    double gflops = 2.0 * (double)n * n * n / (med * 1e-3) / 1e9;
    unsigned long long reads = 2ULL * (unsigned long long)n * n * n;   // 2·n por hilo, n² hilos

    printf("matmul_naive  n=%d  bloque=%dx%d  malla=%dx%d  (%d hilos por bloque)\n",
           n, BLOCK, BLOCK, grid.x, grid.y, BLOCK * BLOCK);
    printf("  lecturas a memoria global (modelo): %llu   (cada elemento de A se lee %d veces)\n",
           reads, n);
    printf("  kernel: mediana %.3f ms (min %.3f, max %.3f) de %d repeticiones -> %.1f GFLOP/s\n",
           med, mn, mx, REPS, gflops);
    printf("  copias: H2D %.3f ms, D2H %.3f ms\n", h2d_ms, d2h_ms);
    printf("  verificacion vs CPU (%ld elementos): error maximo %.2e -> %s\n",
           checked, max_err, ok ? "OK" : "FALLA");
    // Línea para el CSV del notebook: version,n,tile,kernel_med_ms,min,max,h2d_ms,d2h_ms,gflops,max_err,estado
    printf("csv,naive,%d,%d,%.4f,%.4f,%.4f,%.4f,%.4f,%.2f,%.3e,%s\n",
           n, BLOCK, med, mn, mx, h2d_ms, d2h_ms, gflops, max_err, ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaEventDestroy(e_start));
    CUDA_CHECK(cudaEventDestroy(e_stop));
    CUDA_CHECK(cudaFree(d_A)); CUDA_CHECK(cudaFree(d_B)); CUDA_CHECK(cudaFree(d_C));
    free(h_A); free(h_B); free(h_C);
    return ok ? 0 : 1;
}

In [ ]:
%%writefile matmul_tiled.cu
// matmul_tiled.cu — multiplicación de matrices C = A·B (n×n, float, por filas) en CUDA.
// Versión POR MOSAICOS (tiling): cada bloque de TILE×TILE hilos recorre la dimensión k en
// fases; en cada fase copia un mosaico de A y uno de B a memoria COMPARTIDA (__shared__),
// sincroniza, y todos los hilos del bloque reutilizan esos datos TILE veces.
//
// Compilar (GPU):        nvcc -O2 -arch=sm_75 matmul_tiled.cu -o matmul_tiled
// Verificar sin GPU:     nvcc -c -arch=sm_75 matmul_tiled.cu -o matmul_tiled.o
// Ejecutar:              ./matmul_tiled [n]        (n por defecto: 512)
// Tamaño de mosaico:     -DTILE=8 | 16 | 32        (por defecto 16)
// Experimentos (fase Investigar):
//   -DSIN_SYNC1          quita la primera __syncthreads() (carrera sobre el mosaico)
//   -DGRID_TRUNCADO      malla n/TILE en lugar de ceil(n/TILE): elementos sin calcular
#include "common.cuh"

#ifndef TILE
#define TILE 16
#endif

__global__ void matmul_tiled(const float *A, const float *B, float *C, int n) {
    // Memoria compartida del bloque: dos mosaicos de TILE×TILE floats (2·TILE²·4 bytes).
    __shared__ float s_A[TILE][TILE];
    __shared__ float s_B[TILE][TILE];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;      // fila de C que calcula este hilo
    int col = blockIdx.x * TILE + tx;      // columna de C que calcula este hilo
    float acc = 0.0f;

    int n_tiles = (n + TILE - 1) / TILE;   // fases: una por mosaico a lo largo de k
    for (int t = 0; t < n_tiles; ++t) {
        // (1) Carga cooperativa: cada hilo trae UN elemento de A y UNO de B al mosaico.
        //     Los hilos de un warp (tx consecutivos) leen direcciones consecutivas → coalescido.
        int a_col = t * TILE + tx;         // columna de A que lee este hilo
        int b_row = t * TILE + ty;         // fila de B que lee este hilo
        s_A[ty][tx] = (row < n && a_col < n) ? A[row * n + a_col] : 0.0f;   // fuera de rango → 0
        s_B[ty][tx] = (b_row < n && col < n) ? B[b_row * n + col] : 0.0f;   // (n no múltiplo de TILE)
#ifndef SIN_SYNC1
        __syncthreads();   // (2) Barrera: el mosaico completo está en compartida antes de usarlo.
#endif
        // (3) Producto parcial con datos de memoria compartida: TILE reutilizaciones por carga.
        for (int k = 0; k < TILE; ++k) {
            acc += s_A[ty][k] * s_B[k][tx];
        }
        __syncthreads();   // (4) Barrera: nadie sobreescribe el mosaico mientras otros aún lo leen.
    }
    if (row < n && col < n) {              // guardia de escritura
        C[row * n + col] = acc;
    }
}

int main(int argc, char **argv) {
    int n = parse_n(argc, argv, 512);
    size_t bytes = (size_t)n * n * sizeof(float);

    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C = (float *)malloc(bytes);
    if (!h_A || !h_B || !h_C) { fprintf(stderr, "Sin memoria en el anfitrión\n"); return 1; }
    init_matrices(h_A, h_B, n);

    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));

    cudaEvent_t e_start, e_stop;
    CUDA_CHECK(cudaEventCreate(&e_start));
    CUDA_CHECK(cudaEventCreate(&e_stop));

    CUDA_CHECK(cudaEventRecord(e_start));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(e_stop));
    CUDA_CHECK(cudaEventSynchronize(e_stop));
    float h2d_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&h2d_ms, e_start, e_stop));

    CUDA_CHECK(cudaMemset(d_C, 0, bytes));

    dim3 block(TILE, TILE);
#ifdef GRID_TRUNCADO
    int n_blocks = n / TILE;                 // ¡división entera! si n no es múltiplo, faltan bloques
#else
    int n_blocks = (n + TILE - 1) / TILE;    // redondeo hacia arriba: cubre todo n
#endif
    dim3 grid(n_blocks, n_blocks);

    // Calentamiento + revisión de errores. Con TILE=64 el bloque tendría 4096 hilos (> 1024):
    // el lanzamiento falla y SOLO se nota aquí, con cudaGetLastError.
    matmul_tiled<<<grid, block>>>(d_A, d_B, d_C, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    float t_ms[REPS];
    for (int r = 0; r < REPS; ++r) {
        CUDA_CHECK(cudaEventRecord(e_start));
        matmul_tiled<<<grid, block>>>(d_A, d_B, d_C, n);
        CUDA_CHECK(cudaEventRecord(e_stop));
        CUDA_CHECK(cudaEventSynchronize(e_stop));
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventElapsedTime(&t_ms[r], e_start, e_stop));
    }

    CUDA_CHECK(cudaEventRecord(e_start));
    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(e_stop));
    CUDA_CHECK(cudaEventSynchronize(e_stop));
    float d2h_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&d2h_ms, e_start, e_stop));

    float max_err = 0.0f;
    long checked = verify_matmul(h_A, h_B, h_C, n, &max_err);
    int ok = (max_err <= tolerance(n));

    float med = median_of(t_ms, REPS);
    float mn = min_of(t_ms, REPS), mx = max_of(t_ms, REPS);
    double gflops = 2.0 * (double)n * n * n / (med * 1e-3) / 1e9;
    int n_tiles = (n + TILE - 1) / TILE;
    // Modelo: cada hilo lee 2 elementos por fase, n_tiles fases, n² hilos.
    unsigned long long reads = 2ULL * (unsigned long long)n * n * n_tiles;

    printf("matmul_tiled  n=%d  TILE=%d  malla=%dx%d  fases=%d  compartida=%zu bytes por bloque\n",
           n, TILE, grid.x, grid.y, n_tiles, 2 * sizeof(float) * TILE * TILE);
    printf("  lecturas a memoria global (modelo): %llu   (cada elemento de A se lee %d veces)\n",
           reads, n_tiles);
    printf("  kernel: mediana %.3f ms (min %.3f, max %.3f) de %d repeticiones -> %.1f GFLOP/s\n",
           med, mn, mx, REPS, gflops);
    printf("  copias: H2D %.3f ms, D2H %.3f ms\n", h2d_ms, d2h_ms);
    printf("  verificacion vs CPU (%ld elementos): error maximo %.2e -> %s\n",
           checked, max_err, ok ? "OK" : "FALLA");
    printf("csv,tiled,%d,%d,%.4f,%.4f,%.4f,%.4f,%.4f,%.2f,%.3e,%s\n",
           n, TILE, med, mn, mx, h2d_ms, d2h_ms, gflops, max_err, ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaEventDestroy(e_start));
    CUDA_CHECK(cudaEventDestroy(e_stop));
    CUDA_CHECK(cudaFree(d_A)); CUDA_CHECK(cudaFree(d_B)); CUDA_CHECK(cudaFree(d_C));
    free(h_A); free(h_B); free(h_C);
    return ok ? 0 : 1;
}

In [ ]:
%%writefile transpose.cu
// transpose.cu — transposición de una matriz n×n (float, por filas): out = inᵀ.
// Fase CREAR de la práctica guiada. Viene completa la versión INGENUA (lee coalescido,
// escribe saltado). Debes escribir transpose_shared: usa un mosaico en memoria compartida
// para que tanto la lectura como la escritura a memoria global sean coalescidas.
//
// Compilar (GPU):        nvcc -O2 -arch=sm_75 transpose.cu -o transpose
// Verificar sin GPU:     nvcc -c -arch=sm_75 transpose.cu -o transpose.o
// Ejecutar:              ./transpose [n]          (n por defecto: 1024)
// Tamaño de mosaico:     -DTILE=8 | 16 | 32       (por defecto 16)
#include "common.cuh"

#ifndef TILE
#define TILE 16
#endif

// Cambia a 1 cuando termines TODO(1) (o compila con -DTODO1_LISTO=1). Mientras valga 0, el
// programa imprime "TODO pendiente" para la versión con compartida y solo ejecuta (y verifica)
// la ingenua.
#ifndef TODO1_LISTO
#define TODO1_LISTO 0
#endif

// Versión ingenua. Lectura: los hilos de un warp (threadIdx.x consecutivos) leen
// in[row*n + col] con col consecutivos → direcciones consecutivas → coalescido.
// Escritura: out[col*n + row] con col consecutivos → direcciones separadas por n*4 bytes
// → cada hilo toca un segmento distinto de memoria → NO coalescido.
__global__ void transpose_naive(const float *in, float *out, int n) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < n && col < n) {
        out[col * n + row] = in[row * n + col];
    }
}

// Versión con memoria compartida (la escribes tú).
// Idea: el bloque (bx, by) lee el mosaico (by, bx) de `in` POR FILAS y lo deja en `tile`;
// después de una barrera, escribe ese mismo mosaico transpuesto en la posición (bx, by) de
// `out`, también POR FILAS. En los DOS accesos a memoria global, threadIdx.x debe recorrer la
// dirección que cambia más rápido. La transposición ocurre al indexar `tile`, no la global.
// El "+1" en la segunda dimensión evita conflictos de banco en la compartida (tema de S04;
// por ahora solo déjalo).
__global__ void transpose_shared(const float *in, float *out, int n) {
    __shared__ float tile[TILE][TILE + 1];
    /* TODO(1): implementa la transposición por mosaicos.
       Criterio de "listo": ./transpose 1000 y ./transpose 1024 reportan OK en ambas versiones
       y, en la GPU, transpose_shared alcanza más GB/s que transpose_naive para n = 4096. */
    if (n < 0) {   // nunca se cumple (n > 0): solo evita avisos del compilador mientras el TODO
        tile[0][0] = in[0];   // está pendiente. Bórralo al implementar.
        out[0] = tile[0][0];
    }
}

// Referencia secuencial.
static void transpose_cpu(const float *in, float *out, int n) {
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < n; ++j)
            out[(size_t)j * n + i] = in[(size_t)i * n + j];
}

// Mide un kernel de transposición: calentamiento, REPS repeticiones, verificación exacta
// (es una copia: el error debe ser 0). Devuelve 1 si el resultado es correcto.
typedef void (*transpose_kernel_t)(const float *, float *, int);

static int run_and_check(const char *name, transpose_kernel_t kernel, const float *d_in,
                         float *d_out, float *h_out, const float *h_ref, int n) {
    size_t bytes = (size_t)n * n * sizeof(float);
    dim3 block(TILE, TILE);
    dim3 grid((n + TILE - 1) / TILE, (n + TILE - 1) / TILE);
    cudaEvent_t e_start, e_stop;
    CUDA_CHECK(cudaEventCreate(&e_start));
    CUDA_CHECK(cudaEventCreate(&e_stop));

    CUDA_CHECK(cudaMemset(d_out, 0, bytes));
    kernel<<<grid, block>>>(d_in, d_out, n);          // calentamiento
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    float t_ms[REPS];
    for (int r = 0; r < REPS; ++r) {
        CUDA_CHECK(cudaEventRecord(e_start));
        kernel<<<grid, block>>>(d_in, d_out, n);
        CUDA_CHECK(cudaEventRecord(e_stop));
        CUDA_CHECK(cudaEventSynchronize(e_stop));
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventElapsedTime(&t_ms[r], e_start, e_stop));
    }
    CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));

    float max_err = 0.0f;
    for (size_t idx = 0; idx < (size_t)n * n; ++idx) {
        float e = fabsf(h_out[idx] - h_ref[idx]);
        if (e > max_err) max_err = e;
    }
    int ok = (max_err == 0.0f);
    float med = median_of(t_ms, REPS);
    float mn = min_of(t_ms, REPS), mx = max_of(t_ms, REPS);
    // Ancho de banda efectivo: se leen n² floats y se escriben n² floats.
    double gbps = 2.0 * (double)bytes / (med * 1e-3) / 1e9;

    printf("%-16s n=%d TILE=%d  mediana %.3f ms (min %.3f, max %.3f) -> %.1f GB/s  "
           "error maximo %.1e -> %s\n", name, n, TILE, med, mn, mx, gbps, max_err,
           ok ? "OK" : "FALLA");
    printf("csv,%s,%d,%d,%.4f,%.4f,%.4f,0,0,%.2f,%.1e,%s\n", name, n, TILE, med, mn, mx,
           gbps, max_err, ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaEventDestroy(e_start));
    CUDA_CHECK(cudaEventDestroy(e_stop));
    return ok;
}

int main(int argc, char **argv) {
    int n = parse_n(argc, argv, 1024);
    size_t bytes = (size_t)n * n * sizeof(float);

    float *h_in = (float *)malloc(bytes);
    float *h_out = (float *)malloc(bytes);
    float *h_ref = (float *)malloc(bytes);
    if (!h_in || !h_out || !h_ref) { fprintf(stderr, "Sin memoria en el anfitrión\n"); return 1; }
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < n; ++j)
            h_in[(size_t)i * n + j] = (float)((7 * i + 3 * j) % 101);   // exacto en float
    transpose_cpu(h_in, h_ref, n);

    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, bytes));
    CUDA_CHECK(cudaMalloc(&d_out, bytes));
    CUDA_CHECK(cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice));

    int ok = run_and_check("transpose_naive", transpose_naive, d_in, d_out, h_out, h_ref, n);
#if TODO1_LISTO
    ok = run_and_check("transpose_shared", transpose_shared, d_in, d_out, h_out, h_ref, n) && ok;
#else
    printf("TODO pendiente: transpose_shared (TODO(1)); solo se ejecuto la version ingenua.\n");
#endif

    CUDA_CHECK(cudaFree(d_in)); CUDA_CHECK(cudaFree(d_out));
    free(h_in); free(h_out); free(h_ref);
    return ok ? 0 : 1;
}

In [ ]:
%%writefile matmul_cpu.c
/* matmul_cpu.c — referencia secuencial de C = A·B (n×n, float, por filas) y calculadora del
 * modelo de lecturas a memoria global de las dos versiones CUDA (ingenua y por mosaicos).
 * Sirve para: (a) tener una salida real sin GPU, (b) comparar la suma de comprobación con la
 * de la GPU o del simulador de Numba, (c) contrastar la predicción de lecturas de PRIMM.
 *
 * Compilar:  gcc -std=c11 -O2 -Wall -Wextra matmul_cpu.c -o matmul_cpu
 * Ejecutar:  ./matmul_cpu [n] [tile]      (por defecto n=512, tile=16)
 */
#define _POSIX_C_SOURCE 199309L
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

/* Misma inicialización que common.cuh y matmul_numba.py (checksums comparables). */
static void init_matrices(float *A, float *B, int n) {
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < n; ++j) {
            A[(size_t)i * n + j] = (float)((3 * i + 7 * j) % 11) / 11.0f;
            B[(size_t)i * n + j] = (float)((5 * i + 2 * j) % 13) / 13.0f;
        }
    }
}

/* Orden i-k-j: el ciclo interno recorre B y C por filas (amigable con la caché). */
static void matmul_cpu(const float *A, const float *B, float *C, int n) {
    for (size_t idx = 0; idx < (size_t)n * n; ++idx) C[idx] = 0.0f;
    for (int i = 0; i < n; ++i) {
        for (int k = 0; k < n; ++k) {
            float a = A[(size_t)i * n + k];
            for (int j = 0; j < n; ++j) {
                C[(size_t)i * n + j] += a * B[(size_t)k * n + j];
            }
        }
    }
}

static double now_s(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + 1e-9 * (double)ts.tv_nsec;
}

int main(int argc, char **argv) {
    int n = (argc > 1) ? atoi(argv[1]) : 512;
    int tile = (argc > 2) ? atoi(argv[2]) : 16;
    if (n <= 0 || tile <= 0) {
        fprintf(stderr, "Uso: %s [n] [tile]   (ambos > 0)\n", argv[0]);
        return 1;
    }
    size_t count = (size_t)n * n;
    float *A = malloc(count * sizeof(float));
    float *B = malloc(count * sizeof(float));
    float *C = malloc(count * sizeof(float));
    if (!A || !B || !C) { fprintf(stderr, "Sin memoria\n"); return 1; }
    init_matrices(A, B, n);

    double t0 = now_s();
    matmul_cpu(A, B, C, n);
    double t1 = now_s();

    /* Suma de comprobación en doble precisión (misma en matmul_numba.py). */
    double checksum = 0.0;
    for (size_t idx = 0; idx < count; ++idx) checksum += (double)C[idx];

    /* Modelo de lecturas a memoria global (instrucciones de carga, sin contar cachés):
     *   ingenua:  cada hilo lee n elementos de A y n de B → 2n por hilo, n² hilos → 2n³
     *   mosaicos: cada hilo lee 2 elementos por fase, ceil(n/tile) fases → 2·n²·ceil(n/tile)
     */
    int n_tiles = (n + tile - 1) / tile;
    unsigned long long reads_naive = 2ULL * (unsigned long long)n * n * n;
    unsigned long long reads_tiled = 2ULL * (unsigned long long)n * n * n_tiles;

    printf("matmul_cpu  n=%d  tiempo %.4f s  checksum %.6f\n", n, t1 - t0, checksum);
    printf("  modelo de lecturas a memoria global con TILE=%d (fases=%d):\n", tile, n_tiles);
    printf("    ingenua : %llu   (cada elemento de A se lee %d veces)\n", reads_naive, n);
    printf("    mosaicos: %llu   (cada elemento de A se lee %d veces)\n", reads_tiled, n_tiles);
    printf("    cociente ingenua/mosaicos: %.2f\n", (double)reads_naive / (double)reads_tiled);
    printf("  C[0][0]=%.6f  C[n-1][n-1]=%.6f\n", C[0], C[count - 1]);
    printf("csv,cpu,%d,%d,%.4f,%.6f\n", n, tile, (t1 - t0) * 1e3, checksum);

    free(A); free(B); free(C);
    return 0;
}

In [ ]:
%%writefile matmul_numba.py
#!/usr/bin/env python3
"""matmul_numba.py — espejo en Numba CUDA de matmul_naive.cu y matmul_tiled.cu.

Sirve para verificar la lógica de los kernels SIN GPU con el simulador:
    NUMBA_ENABLE_CUDASIM=1 python3 matmul_numba.py --n 64 --tile 16
y también corre en una GPU real (Colab/Kaggle) sin la variable de entorno.

Opciones de la fase Investigar:
    --sin-sync1       quita la primera cuda.syncthreads() del kernel por mosaicos
    --grid-truncado   malla n//tile en lugar de ceil(n/tile)
Las matrices se inicializan con la misma fórmula que common.cuh y matmul_cpu.c, así que la
suma de comprobación (checksum) es comparable entre los tres programas.
"""
import argparse
import os
import sys
import time
import warnings

import numpy as np
from numba import cuda, float32

# Sin la primera barrera se lee memoria compartida no inicializada: puede contener basura
# (NaN, inf). NumPy avisaría en cada operación; silenciamos el aviso para leer la salida.
warnings.filterwarnings("ignore", category=RuntimeWarning)


def init_matrices(n):
    """Misma fórmula que init_matrices() en common.cuh y matmul_cpu.c."""
    i = np.arange(n, dtype=np.int64).reshape(n, 1)
    j = np.arange(n, dtype=np.int64).reshape(1, n)
    A = (((3 * i + 7 * j) % 11) / 11.0).astype(np.float32)
    B = (((5 * i + 2 * j) % 13) / 13.0).astype(np.float32)
    return A, B


def make_kernels(tile, sync1=True):
    """Construye los dos kernels. `tile` debe ser constante en tiempo de compilación porque
    dimensiona los arreglos en memoria compartida; por eso se captura en la clausura."""

    @cuda.jit
    def matmul_naive(A, B, C, n):
        # Un hilo por elemento de C. Mismos índices que en matmul_naive.cu.
        col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
        if row < n and col < n:
            acc = float32(0.0)
            for k in range(n):
                acc += A[row, k] * B[k, col]      # 2 lecturas globales por iteración
            C[row, col] = acc

    @cuda.jit
    def matmul_tiled(A, B, C, n):
        s_A = cuda.shared.array((tile, tile), float32)   # mosaico de A en compartida
        s_B = cuda.shared.array((tile, tile), float32)   # mosaico de B en compartida
        tx = cuda.threadIdx.x
        ty = cuda.threadIdx.y
        row = cuda.blockIdx.y * tile + ty
        col = cuda.blockIdx.x * tile + tx
        acc = float32(0.0)
        n_tiles = (n + tile - 1) // tile
        for t in range(n_tiles):
            # (1) carga cooperativa; fuera de rango se escribe 0 (n no múltiplo de tile)
            a_col = t * tile + tx
            b_row = t * tile + ty
            s_A[ty, tx] = A[row, a_col] if (row < n and a_col < n) else float32(0.0)
            s_B[ty, tx] = B[b_row, col] if (b_row < n and col < n) else float32(0.0)
            if sync1:
                cuda.syncthreads()               # (2) el mosaico completo está en compartida
            for k in range(tile):                # (3) tile reutilizaciones por carga
                acc += s_A[ty, k] * s_B[k, tx]
            cuda.syncthreads()                   # (4) nadie sobreescribe antes de tiempo
        if row < n and col < n:
            C[row, col] = acc

    return matmul_naive, matmul_tiled


def run(kernel, name, A, B, n, tile, grid, reps, reads, reads_per_a, tol, simulado):
    """Lanza el kernel con calentamiento + reps repeticiones; verifica contra NumPy."""
    C = np.zeros((n, n), dtype=np.float32)       # ceros: un elemento no calculado se nota
    block = (tile, tile)
    if not simulado:                             # en el simulador el calentamiento no aporta
        kernel[grid, block](A, B, C, n)          # calentamiento (y compilación JIT)
        cuda.synchronize()
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        kernel[grid, block](A, B, C, n)
        cuda.synchronize()
        times.append(time.perf_counter() - t0)
    median = sorted(times)[len(times) // 2]
    ref = A.astype(np.float64) @ B.astype(np.float64)
    max_err = float(np.max(np.abs(C.astype(np.float64) - ref)))
    ok = np.isfinite(max_err) and max_err <= tol
    print(f"{name:<9} n={n} tile={tile} malla={grid[0]}x{grid[1]} bloque={tile}x{tile}  "
          f"lecturas_globales(modelo)={reads}  (cada elemento de A: {reads_per_a} veces)")
    print(f"          error maximo vs NumPy {max_err:.2e}  mediana {median*1e3:.3f} ms "
          f"de {reps} -> {'PASS' if ok else 'FAIL'}")
    return C, ok


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--n", type=int, default=64, help="tamaño n de las matrices n×n")
    parser.add_argument("--tile", type=int, default=16, help="tamaño del mosaico/bloque")
    parser.add_argument("--reps", type=int, default=1, help="repeticiones cronometradas (GPU: 5)")
    parser.add_argument("--sin-sync1", action="store_true", help="quita la 1a syncthreads")
    parser.add_argument("--grid-truncado", action="store_true", help="malla n//tile")
    parser.add_argument("--solo", choices=["ingenua", "mosaicos"], default=None,
                        help="ejecuta solo una versión")
    args = parser.parse_args()
    n, tile = args.n, args.tile
    if n <= 0 or tile <= 0:
        sys.exit("n y tile deben ser > 0")

    simulado = os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"
    if simulado:
        print("[CUDASIM] simulador de CUDA activo: se verifica la logica; "
              "los tiempos NO son representativos")

    A, B = init_matrices(n)
    n_tiles = (n + tile - 1) // tile
    blocks = n // tile if args.grid_truncado else n_tiles
    grid = (blocks, blocks)
    if args.grid_truncado:
        print(f"[grid-truncado] malla {blocks}x{blocks} con division entera "
              f"(deberia ser {n_tiles}x{n_tiles})")
    if args.sin_sync1:
        print("[sin-sync1] se quito la primera cuda.syncthreads() del kernel por mosaicos")

    naive, tiled = make_kernels(tile, sync1=not args.sin_sync1)
    tol = 1e-4 * n                               # misma tolerancia que common.cuh
    all_ok = True
    C = None
    if args.solo != "mosaicos":
        C, ok = run(naive, "ingenua", A, B, n, tile, grid, args.reps,
                    reads=2 * n * n * n, reads_per_a=n, tol=tol, simulado=simulado)
        all_ok = all_ok and ok
    if args.solo != "ingenua":
        C, ok = run(tiled, "mosaicos", A, B, n, tile, grid, args.reps,
                    reads=2 * n * n * n_tiles, reads_per_a=n_tiles, tol=tol, simulado=simulado)
        all_ok = all_ok and ok

    checksum = float(np.sum(C, dtype=np.float64))
    print(f"checksum C = {checksum:.6f}   (compara con ./matmul_cpu {n} {tile})")
    print(f"csv,numba,{n},{tile},{'sim' if simulado else 'gpu'},{checksum:.6f},"
          f"{'PASS' if all_ok else 'FAIL'}")
    sys.exit(0 if all_ok else 1)


if __name__ == "__main__":
    main()

## Paso 2 · Compilar

`-arch=sm_75` es la arquitectura de la T4 (Turing). Si te asignan otra GPU, el PTX incluido se recompila al vuelo.

In [ ]:
!nvcc -O2 -arch=sm_75 matmul_naive.cu -o matmul_naive
!nvcc -O2 -arch=sm_75 matmul_tiled.cu -o matmul_tiled
!nvcc -O2 -arch=sm_75 transpose.cu -o transpose
!gcc -std=c11 -O2 -Wall -Wextra matmul_cpu.c -o matmul_cpu
!ls -l matmul_naive matmul_tiled transpose matmul_cpu

## Paso 3 · Ejecutar (fase **Ejecutar** de PRIMM)

Primero `n = 64`, el caso de tus predicciones: compara las lecturas del modelo y el *checksum* de la CPU con la GPU. Después `n = 512`.

In [ ]:
!./matmul_cpu 64 16
!./matmul_naive 64
!./matmul_tiled 64

In [ ]:
!./matmul_naive 512
!./matmul_tiled 512

## Paso 4 · Investigar

**4a.** `n` no múltiplo de `TILE`: ¿cuántas fases y cuántos bloques? ¿la verificación sigue en OK?

In [ ]:
!./matmul_tiled 100
!./matmul_tiled 20
!./matmul_tiled 1000

**4b.** Dos programas que **no reportan ningún error de CUDA** y aun así están mal. Corre la versión sin la primera barrera tres veces: ¿sale lo mismo?

In [ ]:
!nvcc -O2 -arch=sm_75 -DSIN_SYNC1 matmul_tiled.cu -o matmul_tiled_sin_sync1
!for i in 1 2 3; do ./matmul_tiled_sin_sync1 512 | grep -E "verificacion|csv"; done
!nvcc -O2 -arch=sm_75 -DGRID_TRUNCADO matmul_tiled.cu -o matmul_tiled_grid_truncado
!./matmul_tiled_grid_truncado 100

**4c.** El espejo en Numba corre también en la GPU real (sin `NUMBA_ENABLE_CUDASIM`). Misma lógica, mismo *checksum*.

In [ ]:
!python3 matmul_numba.py --n 512 --tile 16 --reps 5

## Paso 5 · Modificar: tamaño de mosaico y de bloque

Compila `TILE` = 8, 16, 32 (y `BLOCK` = 8, 16, 32 para la ingenua). Después intenta `TILE = 64`: ¿compila? ¿corre? ¿quién lo detecta?

In [ ]:
%%bash
for t in 8 16 32; do
  nvcc -O2 -arch=sm_75 -DTILE=$t  matmul_tiled.cu -o matmul_tiled_t$t
  nvcc -O2 -arch=sm_75 -DBLOCK=$t matmul_naive.cu -o matmul_naive_b$t
done
nvcc -O2 -arch=sm_75 -DTILE=64 matmul_tiled.cu -o matmul_tiled_t64 && echo "TILE=64 compila"
./matmul_tiled_t64 512; echo "codigo de salida con TILE=64: $?"

## Paso 6 · Medición con el protocolo del curso

Cada programa ya hace calentamiento + 5 repeticiones y reporta la mediana; aquí se recorren `n` y `TILE` y se recogen las líneas `csv,...`. Tarda 1–3 minutos en una T4.

In [ ]:
import datetime, subprocess
import pandas as pd

rows = []
COLS = ["version", "n", "tile", "kernel_med_ms", "kernel_min_ms", "kernel_max_ms",
        "h2d_ms", "d2h_ms", "gflops_o_gbps", "max_err", "estado"]

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for line in out.stdout.splitlines():
        if line.startswith("csv,"):
            f = line.split(",")[1:]
            rows.append(dict(zip(COLS, [f[0], int(f[1]), int(f[2])] + [float(x) for x in f[3:10]] + [f[10]])))
    if out.returncode != 0:
        print("AVISO: código", out.returncode, "en", cmd, "|", out.stderr.strip()[-200:])

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
for n in [256, 512, 1024, 2048]:
    for t in [8, 16, 32]:
        run(f"./matmul_naive_b{t} {n}")
        run(f"./matmul_tiled_t{t} {n}")
for n in [1024, 2048, 4096]:
    run(f"./transpose {n}")

df = pd.DataFrame(rows)
df["gpu"] = gpu
df["fecha"] = datetime.date.today().isoformat()
print("GPU:", gpu)
print(df[df.version.isin(["naive", "tiled"])]
        .pivot_table(index="n", columns=["version", "tile"], values="kernel_med_ms").round(3))
print(df[df.version.str.startswith("transpose")][["version", "n", "kernel_med_ms", "gflops_o_gbps", "estado"]])

## Paso 7 · Crear: transposición coalescida con memoria compartida

Completa `TODO(1)` en `transpose.cu` (celda `%%writefile transpose.cu` arriba), cambia `TODO1_LISTO` a 1, vuelve a ejecutar esa celda y compila aquí. Criterio de listo: `OK` en `n = 1000` y `n = 1024`, y más GB/s que la ingenua en `n = 4096`.

In [ ]:
!nvcc -O2 -arch=sm_75 transpose.cu -o transpose
!./transpose 1000
!./transpose 1024
!./transpose 4096

## Paso 8 · Guardar los resultados (CSV)

El archivo va a `resultados/` del repositorio de tu equipo (Práctica 6, Parte B/D) junto con la GPU y la fecha.

In [ ]:
nombre = f"resultados_S02_matmul_{datetime.date.today().isoformat()}.csv"
df.to_csv(nombre, index=False)
print(nombre, "->", len(df), "filas")
print(df.head(8).to_string())
try:
    from google.colab import files
    files.download(nombre)
except Exception as exc:  # fuera de Colab (Kaggle): descarga manual desde el panel de archivos
    print("Descarga manual desde el panel de archivos:", type(exc).__name__)

## Qué te llevas

1. La tabla `n × TILE` de tiempos (naive vs tiled) y la razón de GFLOP/s entre ambas para `n = 1024`.
2. Tu respuesta, con los números de arriba, a: *¿por qué TILE=32 no es 2× más rápido que TILE=16 si hace la mitad de lecturas globales?*
3. La salida de 4b: dos programas sin error de CUDA y con resultado incorrecto. Es tu evidencia para la bitácora de errores de la Práctica 6.
4. El CSV con GPU y fecha.